# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset on human mast cell extracellular vesicle proteomics using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Let's review the available record sets, fields, and their `@id` values. Each record set contains fields that define the structure of the data, and all entities are referenced by their unique `@id`.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets defined in metadata.")
else:
    for rs in record_sets:
        print(f"Record set: {rs['@id']}  (name: {rs.get('name', 'N/A')})")
        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                field_id = field['@id'] if isinstance(field, dict) else str(field)
                print(f"    - {field_id}")

Let's print the first few records from a specific record set. Choose the record set `@id` from the output above to continue exploration. *(Below we use the main experiment data record set as example: 'dv:ExperimentDataset')*

In [ ]:
# Print a few records from a chosen record set (@id: 'dv:ExperimentDataset')
example_record_set_id = 'dv:ExperimentDataset'

try:
    for idx, rec in enumerate(dataset.records(record_set=example_record_set_id)):
        if idx >= 3:
            break
        print(f"Record {idx+1}:")
        for k, v in rec.items():
            print(f"  {k}: {v}")
except Exception as e:
    print(f"Error loading records from {example_record_set_id}: {e}")

## 3. Data Extraction
Load data from available record sets into pandas DataFrames for analysis. Use the `@id` values obtained earlier.
We'll demonstrate with 'dv:ExperimentDataset' as the main data table.

In [ ]:
# Extract data from defined record sets
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets] if dataset.metadata.record_sets else ['dv:ExperimentDataset']
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for: {record_set_id} | Columns: {dataframes[record_set_id].columns.tolist()}")
        else:
            print(f"No records found for {record_set_id}.")
    except Exception as e:
        print(f"Error loading records from {record_set_id}: {e}")

# Preview the first few rows of the main table
main_rs_id = 'dv:ExperimentDataset'
if main_rs_id in dataframes:
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping.

*We use the column `dv:Field:MW` (presumed molecular weight) and the `dv:Field:description` (protein description) if present. Otherwise, adapt as necessary based on available columns printed above. All column access is by `@id`.*

In [ ]:
# Identify numeric and grouping fields by @id
main_df = dataframes.get(main_rs_id)

# Check available columns
if main_df is not None and len(main_df.columns) > 0:
    print(f"Available columns (@id): {main_df.columns.tolist()}")

    # Try with known field @ids (adjust as needed):
    numeric_field_id = 'dv:Field:MW' if 'dv:Field:MW' in main_df.columns else main_df.select_dtypes('number').columns[0]
    group_field_id = 'dv:Field:description' if 'dv:Field:description' in main_df.columns else main_df.columns[0]

    print(f"Numeric field (@id): {numeric_field_id}")
    print(f"Group field (@id): {group_field_id}")

    # Filter records with molecular weight > 10000
    threshold = 10000
    filtered_df = main_df[main_df[numeric_field_id].astype(float) > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id and calculate mean
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No suitable DataFrame or columns available for EDA.")

## 5. Visualization
Visualize the distribution of the `@id`-referenced numeric field (e.g., molecular weight) or a relationship between two fields (with `@id` references).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of molecular weights
if main_df is not None and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Pairplot with two fields if available
possible_fields = list(main_df.columns)
if len(possible_fields) > 1:
    field_x = numeric_field_id
    field_y = possible_fields[1]
    try:
        plt.figure(figsize=(7,5))
        sns.scatterplot(x=main_df[field_x].astype(float), y=main_df[field_y])
        plt.xlabel(field_x)
        plt.ylabel(field_y)
        plt.title(f"{field_x} vs {field_y}")
        plt.show()
    except Exception as e:
        print(f"Unable to plot scatter: {e}")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a FAIR² Croissant-formatted dataset using the `mlcroissant` library. We performed exploratory data analysis, normalization, and visualizations, referencing all fields and record sets by their `@id` as specified in the schema. 

This workflow is easily extensible to more advanced analyses using the same approach, ensuring robust, metadata-driven handling of FAIR datasets.